<div align="center">

# FUNDACION UNIVERSITARIA LOS LIBERTADORES

---

## Inteligencia Artificial II
## Proyecto final: Sistema experto con interfaz web

---

# Sistema experto para el diagnóstico de fallas en workflows de n8n

*Troubleshooting de procesos de automatización mediante lógica de predicados, reglas de producción y encadenamiento hacia adelante con factores de certeza*

---

| | |
|---|---|
| **Autores** | Laura Daniela Lopez Santos · Juan Jose Narvaez Ortiz |
| **Programa** | Ingeniería de Sistemas |
| **Asignatura** | Inteligencia Artificial II |
| **Docente** | Gonazalo Ricardo Novoa |
| **Fecha original (Actividad 4)** | 29 de octubre de 2026 |
| **Actualización (proyecto final)** | 8 de noviembre de 2026 |
| **Fase cubierta en este notebook** | Backend: base de conocimiento, motor de inferencia, explicación y validación. La interfaz web (Streamlit) es la siguiente fase |

</div>

## Contenido

1. [Descripción del proyecto](#1.-Descripción-del-proyecto)
2. [Introducción y definición del dominio](#2.-Introducción-y-definición-del-dominio)
3. [Requerimientos](#3.-Requerimientos)
4. [Formalización del conocimiento](#4.-Formalización-del-conocimiento)
5. [Implementación](#5.-Implementación)
6. [Casos de prueba y validación](#6.-Casos-de-prueba-y-validación)
7. [Análisis y conclusiones](#7.-Análisis-y-conclusiones)
8. [Estructura del proyecto y siguientes pasos](#8.-Estructura-del-proyecto-y-siguientes-pasos)


---
# 1. Descripción del proyecto

## 1.1 Descripción general

Este proyecto construye un **sistema experto integral** para el diagnóstico de fallas en workflows de n8n. Un sistema experto captura conocimiento especializado y lo usa para resolver problemas mediante razonamiento automatizado: aquí, a partir de los síntomas y del contexto que reporta el usuario, el sistema **deriva la causa raíz probable, recomienda una acción correctiva y explica qué reglas utilizó** para llegar a esa conclusión.

El proyecto se compone de cuatro elementos:

| Componente | Función | Dónde está |
|---|---|---|
| **Base de conocimiento** | 8 predicados, 15 reglas de producción SI-ENTONCES con factores de certeza y un catálogo de soluciones | Secciones 4 y 5.1 (futuro `base_conocimiento.py`) |
| **Motor de inferencia** | Encadenamiento hacia adelante con factores de certeza, protección contra ciclos, logging y validación de la entrada | Sección 5.2 (futuro `motor_inferencia.py`) |
| **Sistema de explicación** | `explicar_razonamiento()` (todo el razonamiento) y `porque()` (cadena que justifica una conclusión) | Sección 5.2 |
| **Interfaz web** | Aplicación en **Streamlit** para que un usuario no técnico ingrese hechos, vea el diagnóstico y consulte la explicación | **Pendiente** (futuro `app.py`) |

## 1.2 Objetivos

| # | Objetivo específico de la guía | Estado |
|---|---|---|
| 1 | Base de conocimiento con 12-15 reglas formalizadas en lógica de predicados | ✅ 15 reglas (secciones 4 y 5.1) |
| 2 | Motor de inferencia con encadenamiento hacia adelante y explicación del razonamiento | ✅ Sección 5.2 |
| 3 | Interfaz web interactiva para usuarios finales | ⏳ Siguiente fase (Streamlit) |
| 4 | Validación con casos de prueba exhaustivos y documentación del proceso | ✅ Validación (sección 6) · ⏳ documentación técnica y manual de usuario |
| 5 | Integración backend + frontend en una aplicación funcional | ⏳ Siguiente fase |

## 1.3 Qué agrega este proyecto respecto a la actividad anterior

| Aspecto | Actividad 4 (notebook anterior) | Proyecto final |
|---|---|---|
| Reglas | 12 | **15** (se amplía con R13 a R15: permisos 403 y timeout por volumen) |
| Casos de prueba | 6, todos con conclusión | **17**: funcionales, uno de fallas simultáneas, casos sin conclusión, de borde y de ciclo |
| Traducción lógica → código | R3 y R12 se resolvían con atajos (`ErrorHTTP_5xx`, `(CausaRaiz,)`) | **Unificación con variables y restricciones**: R3 usa `c ≥ 500` y R12 es una regla normal del motor |
| Explicación | Imprimía en pantalla; `porque()` solo mostraba la regla directa | Las funciones **retornan texto**; `porque()` recorre la **cadena completa** hasta los hechos observados |
| Casos especiales | Sin estado ni pruebas | Estados `diagnostico_completo`, `diagnostico_parcial`, `sin_conclusion`, `entrada_invalida`; detección de ciclos; logging |
| Entrada | Sin validación | `validar_hechos()` con errores y advertencias |
| Organización del código | Funciones sueltas que imprimen | Celdas organizadas por módulo, **separando lógica y presentación**, listas para pasar a `src/` |
| Interfaz | Ninguna | Streamlit (siguiente fase) |
| Documentación | Notebook | Documentación técnica, manual de usuario y README (siguiente fase) |

## 1.4 Arquitectura general

```
   Usuario ──► [ Interfaz web (Streamlit) ]                        ← pendiente
                      │  hechos observados: códigos HTTP, autenticación, síntomas, contexto
                      ▼
               validar_hechos()              validación básica de la entrada
                      ▼
               diagnosticar() ───────────►  encadenamiento_adelante()
                      │                            ▲
                      │                            │  reglas R1–R15 + catálogo Solucion
                      │                      BASE_CONOCIMIENTO
                      ▼
   causas raíz + recomendaciones + factores de certeza + explicación
        (explicar_razonamiento, porque)
                      ▼
          [ Interfaz web: conclusiones, reglas aplicadas, botón «¿Por qué?» ]
```

**Flujo de datos:** (1) el usuario marca los hechos observados; (2) se validan; (3) el motor aplica las reglas hasta que no surgen conclusiones nuevas; (4) `diagnosticar()` devuelve un diccionario con las causas, las recomendaciones, los factores de certeza y la explicación; (5) la interfaz lo presenta. `diagnosticar()` no imprime nada, de modo que la misma función sirve para el notebook, las pruebas y la interfaz.


---
# 2. Introducción y definición del dominio

## 2.1 Dominio elegido

**Troubleshooting de workflows en n8n** (categoría: *diagnóstico técnico / troubleshooting de procesos*).

n8n es una plataforma de automatización que conecta servicios (APIs, webhooks, bases de datos, CRMs) mediante *workflows* formados por nodos. El sistema recibe como entrada **hechos** sobre un workflow que falla: síntomas (códigos de error HTTP, timeouts, nodos sin datos, webhooks que no disparan) y contexto del nodo (tipo de autenticación, volumen de datos). En esta versión **los hechos los selecciona el usuario** en un formulario; el sistema no lee automáticamente las ejecuciones de n8n. Como validación complementaria, los casos de la Actividad 4 se reprodujeron en un workflow de n8n con nodos reales, donde la mayoría de los hechos se extrae de la ejecución (material aparte). A partir de los hechos, el sistema infiere la **causa raíz probable** y recomienda la **solución**, explicando qué reglas utilizó.

## 2.2 Descripción del problema

Los workflows de n8n integran muchos servicios. Cuando uno falla, el mensaje de error suele ser poco claro y **una misma falla puede tener varias causas**. Quien diagnostica necesita experiencia previa: saber, por ejemplo, que un 401 con OAuth2 suele deberse a un token vencido, o que un webhook que "no recibe nada" suele deberse a un workflow inactivo o a la URL de prueba usada en producción.

Este sistema captura ese conocimiento experto en reglas **SI-ENTONCES**. Dado un conjunto de síntomas, **deduce la causa raíz y recomienda la acción correctiva**.

## 2.3 Justificación de relevancia

- **Agiliza la depuración:** organiza síntomas y contexto en una ruta de diagnóstico, evitando que el usuario tenga que recordar desde cero todas las causas posibles.
- **Hace explícito y reutilizable el conocimiento experto:** las reglas SI-ENTONCES pueden revisarse, explicarse y ampliarse sin ocultar el razonamiento en código opaco.
- **Orienta distintos niveles de experiencia:** ofrece a principiantes una lista de comprobación y permite a expertos contrastar rápidamente hipótesis y acciones correctivas.
- **Demuestra las partes de un sistema basado en conocimiento:** hechos, predicados, reglas, inferencia por Modus Ponens, recomendaciones y explicación de las reglas activadas.
- **Permite contrastar las reglas con evidencia reproducible:** los antecedentes observables (respuestas HTTP 401/403/429/5xx, estado del workflow, URL del webhook, salida de filtros) se pueden provocar y registrar en una instancia de n8n. Las reglas son heurísticas del dominio; no garantizan por sí solas que cada causa se reproduzca en todas las versiones, integraciones o configuraciones.
- **Representa incertidumbre de forma explícita:** un HTTP 401 con OAuth2 es compatible con un token vencido, pero también pueden existir otras causas. Los factores de certeza expresan la confianza asignada a cada inferencia; no son una probabilidad estadística ni sustituyen la verificación de credenciales.

Por ello, el sistema se plantea como apoyo al diagnóstico y no como detector automático ni como sustituto de la inspección del workflow. Su utilidad depende de que los hechos ingresados sean correctos y de que las reglas se revisen frente a cambios de n8n y de los servicios conectados.


## 2.4 Alcance del sistema

### Dentro del alcance

| Área | Casos cubiertos |
|---|---|
| Errores de autenticación | HTTP 401 con OAuth2 (R4) o con API Key (R5) |
| Permisos | HTTP 403 con OAuth2 (R13 y R14, ampliación) |
| Límites de uso | HTTP 429 combinado con volumen alto de items |
| Fallas del servicio externo | HTTP 5xx **con** timeout (R7). Un 5xx sin timeout solo se clasifica como categoría de falla, sin causa raíz |
| Volumen y tiempos de espera | Timeout con muchos items (R15, ampliación) |
| Activación y webhooks | Webhook sin ejecuciones por workflow inactivo o por usar la URL de prueba |
| Flujo de datos | Nodos sin datos por filtros restrictivos |
| Expresiones | Referencias a campos inexistentes |

### Fuera del alcance

- Problemas de infraestructura de la instancia (Docker, base de datos de n8n, memoria, *queue mode*, *reverse proxy*).
- Errores de lógica en el código de nodos *Code* escritos por el usuario.
- Fallas internas del servicio externo: el sistema no verifica su estado real; solo infiere saturación a partir de un 5xx con timeout.
- Otros códigos HTTP como categorías de falla (400, 404, 403 con API Key): quedan como **extensión futura**. El sistema responde "sin conclusión" ante ellos.
- Detección automática de síntomas: los hechos los ingresa el usuario. No se lee el historial de ejecuciones mediante la API de n8n.
- Aspectos de seguridad y auditoría de credenciales.


---
# 3. Requerimientos

Matriz de cumplimiento frente a la guía de desarrollo. Este notebook cubre las fases de **backend y validación**; la interfaz, la estructura de carpetas y la documentación final se desarrollan en las siguientes fases.

| # | Requerimiento de la guía | Estado | Dónde se cumple |
|---|---|---|---|
| | **1. Base de conocimiento** | | |
| 1.1 | 8-12 predicados con tipo, dominio y rango | ✅ | 8 predicados (secciones 4.1 y 4.2); `PREDICADOS` y `DOMINIOS` en el código |
| 1.2 | Documentación en notación lógica matemática | ✅ | Secciones 4.2 y 4.3 |
| 1.3 | 12-15 reglas formalizadas en lógica de predicados | ✅ | 15 reglas (sección 4.3) |
| 1.4 | Traducción a SI-ENTONCES con factores de certeza | ✅ | Secciones 4.4 y 4.5; lista `REGLAS` |
| 1.5 | Organización por categorías o secuencia | ✅ | 4 grupos (campo `grupo` de cada regla) |
| 1.6 | Estructura `PREDICADOS` / `REGLAS` con `nombre`, `descripcion`, `condiciones`, `conclusion`, `certeza` | ✅ | Sección 5.1 |
| | **2. Motor de inferencia** | | |
| 2.1 | `encadenamiento_adelante(base, hechos)` → `(hechos_finales, reglas_aplicadas)` | ✅ | Sección 5.2 |
| 2.2 | `explicar_razonamiento(reglas_aplicadas)` → texto | ✅ | Sección 5.2 |
| 2.3 | `porque(conclusion, hechos_iniciales, reglas_aplicadas)` → cadena de razonamiento | ✅ | Sección 5.2 |
| 2.4 | Código modular y reutilizable | 🔶 Parcial | Celdas separadas por módulo; falta extraerlas a `src/` (sección 8) |
| 2.5 | Manejo de casos sin conclusión | ✅ | Estado `sin_conclusion` y mensaje; casos 12 y 13 |
| 2.6 | Detección de ciclos infinitos | ✅ | `detectar_ciclos()`, límite de iteraciones y control de disparos repetidos; caso 17 |
| 2.7 | Logging del proceso de inferencia | ✅ | Módulo `logging` (logger `sbc.motor`) |
| | **3. Interfaz web (Streamlit)** | | |
| 3.1 | Entrada: formulario con validación básica | ⏳ | El backend ya ofrece `validar_hechos()`, `PREDICADOS` y `DOMINIOS` |
| 3.2 | Resultados: conclusiones destacadas, recomendaciones, certeza | ⏳ | `diagnosticar()` ya devuelve `causas` con recomendación y CF |
| 3.3 | Explicación: reglas en orden, cadena de razonamiento, botón «¿Por qué?» | ⏳ | `explicar_razonamiento()` y `porque()` |
| 3.4 | Navegación: limpiar/resetear y casos precargados | ⏳ | `CASOS_PRUEBA` ya está listo para precargarse |
| | **4. Casos de prueba y validación** | | |
| 4.1 | Mínimo 5 casos en una tabla (caso, descripción, hechos, reglas esperadas, conclusión esperada, resultado obtenido, estado) | ✅ | 17 casos (sección 6) |
| 4.2 | Cada caso ejercita partes distintas de la base de conocimiento | ✅ | Matriz de cobertura (sección 6.3): las 15 reglas |
| 4.3 | Al menos un caso sin conclusión | ✅ | Casos 12 y 13 |
| 4.4 | Al menos un caso que active la mayoría de las reglas | ✅ | Caso 11 (14 de 15 reglas) |
| 4.5 | Casos de borde documentados | ✅ | Casos 12 a 17 |
| | **5. Estructura, documentación y entrega** | | |
| 5 | Estructura de carpetas del proyecto | ⏳ | Sección 8 |
| 6.1 | Documentación técnica (PDF) | ⏳ | Las secciones 1, 2, 4, 5, 6 y 7 de este notebook son su base |
| 6.2 | Manual de usuario de 4-5 páginas con capturas | ⏳ | Requiere la interfaz terminada |
| E | README, `requirements.txt`, casos de prueba documentados y ZIP `lopez_narvaez_proyecto_final_sbc.zip` | ⏳ | Sección 8 |
| S | Demo de 5-7 minutos y sustentación individual | ⏳ | Después de la integración |

Leyenda: ✅ cumplido · 🔶 parcial · ⏳ pendiente.


---
# 4. Formalización del conocimiento

## 4.1 Universos de discurso

Un predicado recibe objetos y devuelve verdadero o falso. El **universo de discurso** es el conjunto de objetos que pueden ocupar cada argumento.

| Símbolo | Conjunto | Ejemplos |
|---|---|---|
| **N** | Nodos del workflow | `n1`, `n2` |
| **W** | Workflows | `w1`, `w2` |
| **X** | N ∪ W (objetos diagnosticables) | `n1`, `w1` |
| **K** | Códigos HTTP: {100, …, 599} | 401, 429, 503 |
| **T** | Tipos de autenticación | `OAuth2`, `ApiKey` |
| **SIN** | Síntomas observables | `TimeoutNodo`, `DatosVacios`, `ExpresionInvalida`, `WebhookSinEjecuciones` |
| **CTX** | Factores de contexto | `ProcesaMuchosItems`, `WorkflowInactivo`, `UsaURLTest`, `FiltroPrevioDescartaTodo` |
| **F** | Categorías de falla | `CredencialInvalida`, `LimiteRateExcedido`, `ErrorServidorExterno`, `PermisosInsuficientes` |
| **C** | Causas raíz posibles | `TokenVencido`, `ApiKeyIncorrecta`, `WorkflowDesactivado`, `ScopesInsuficientes`, `TimeoutPorVolumen`, … |
| **S** | Soluciones posibles | `ReautenticarCredencialOAuth2`, `ActivarElWorkflow`, `ReautorizarCredencialConScopesCorrectos`, … |

> **Nota sobre el "Tipo":** el ejemplo del enunciado llama "binario" a un predicado cuyo valor es verdadero o falso. Para evitar ambigüedad, aquí se indica la **aridad** (número de argumentos) y el **Rango** siempre es booleano.

## 4.2 Predicados del dominio (8)

Se definen **8 predicados**, el mínimo exigido. Para lograrlo con 15 reglas, los síntomas, los factores de contexto y las categorías de falla se modelan como **predicados con un segundo argumento** que toma valores de un conjunto cerrado (SIN, CTX, F). Por ejemplo, en lugar de un predicado `TimeoutNodo(n)` se escribe `Sintoma(n, TimeoutNodo)`.

Los predicados se organizan en tres capas: lo que se **observa**, lo que se **infiere como categoría** y lo que se **concluye y recomienda**.

> **Nota de implementación:** en el código, los hechos omiten el primer argumento (el nodo `n` o workflow `w`) porque el sistema **diagnostica un objeto a la vez**: el objeto bajo análisis queda implícito. Por eso `ErrorHTTP(n, 401)` se escribe `("ErrorHTTP", 401)`. El cuantificador universal ∀ de las reglas se implementa con **variables** (`"?c"`) que el motor liga a los valores de los hechos.

### Grupo A: Predicados de observación (4): lo que el usuario ingresa

```
ErrorHTTP(n, c): El nodo n respondió con el código de error HTTP c
  Tipo: Predicado de aridad 2
  Dominio: n ∈ N, c ∈ K
  Rango: {verdadero, falso}

Autenticacion(n, t): El nodo n se autentica mediante el mecanismo t
  Tipo: Predicado de aridad 2
  Dominio: n ∈ N, t ∈ T = {OAuth2, ApiKey}
  Rango: {verdadero, falso}

Sintoma(x, s): El objeto x presenta el síntoma s
  Tipo: Predicado de aridad 2
  Dominio: x ∈ X, s ∈ SIN = {WebhookSinEjecuciones, DatosVacios,
                             ExpresionInvalida, TimeoutNodo}
  Rango: {verdadero, falso}

Contexto(x, k): El objeto x cumple la condición de contexto k
  Tipo: Predicado de aridad 2
  Dominio: x ∈ X, k ∈ CTX = {ProcesaMuchosItems, WorkflowInactivo,
                             UsaURLTest, FiltroPrevioDescartaTodo}
  Rango: {verdadero, falso}
```

**Significado de los valores de SIN y CTX**

| Constante | Conjunto | Significado |
|---|---|---|
| `WebhookSinEjecuciones` | SIN | El workflow no registra ejecuciones aunque se esperan llamadas por webhook |
| `DatosVacios` | SIN | El nodo recibe o produce cero items |
| `ExpresionInvalida` | SIN | El nodo tiene una expresión que referencia un campo inexistente o con sintaxis errónea |
| `TimeoutNodo` | SIN | El nodo excedió el tiempo máximo de espera |
| `ProcesaMuchosItems` | CTX | El nodo procesa un volumen alto de items (p. ej. más de 100) sin pausas entre solicitudes |
| `WorkflowInactivo` | CTX | El workflow tiene apagado el toggle "Active" |
| `UsaURLTest` | CTX | El servicio externo usa la URL `/webhook-test/` en lugar de `/webhook/` |
| `FiltroPrevioDescartaTodo` | CTX | Un nodo IF o Filter anterior descarta todos los items |

### Grupo B: Predicado intermedio (1): lo que el sistema deriva

```
CategoriaFalla(n, f): La falla del nodo n pertenece a la categoría general f
  Tipo: Predicado de aridad 2
  Dominio: n ∈ N, f ∈ F = {CredencialInvalida, LimiteRateExcedido, ErrorServidorExterno,
                                   PermisosInsuficientes}
  Rango: {verdadero, falso}
```

| Constante | Significado |
|---|---|
| `CredencialInvalida` | El servicio rechazó la credencial del nodo |
| `LimiteRateExcedido` | El nodo superó el límite de solicitudes de la API |
| `ErrorServidorExterno` | La falla se origina en el servidor del servicio externo |
| `PermisosInsuficientes` | El servicio reconoce la credencial, pero niega el permiso solicitado |

### Grupo C: Predicados de diagnóstico y recomendación (3): la salida

```
CausaRaiz(x, c): La causa probable de la falla del objeto x es c
  Tipo: Predicado de aridad 2
  Dominio: x ∈ X, c ∈ C
  Rango: {verdadero, falso}

Solucion(c, s): La acción s corrige la causa c (catálogo de conocimiento fijo)
  Tipo: Predicado de aridad 2
  Dominio: c ∈ C, s ∈ S
  Rango: {verdadero, falso}

Recomendar(x, s): Se recomienda aplicar la acción s sobre el objeto x
  Tipo: Predicado de aridad 2
  Dominio: x ∈ X, s ∈ S
  Rango: {verdadero, falso}
```

**Resumen:** 4 de observación + 1 intermedio + 3 de diagnóstico = **8 predicados**. Cada uno aparece en al menos una regla.

| Predicado | Capa | Reglas donde aparece |
|---|---|---|
| `ErrorHTTP` | A | R1, R2, R3, R13 |
| `Autenticacion` | A | R4, R5, R14 |
| `Sintoma` | A | R7, R8, R9, R10, R11, R15 |
| `Contexto` | A | R6, R8, R9, R10, R15 |
| `CategoriaFalla` | B | R1–R7, R13, R14 |
| `CausaRaiz` | C | R4–R12, R14, R15 |
| `Solucion` | C | R12 |
| `Recomendar` | C | R12 |

## 4.3 Reglas en lógica de predicados (15 reglas)

Las reglas R1 a R12 provienen de la Actividad 4. **R13, R14 y R15 son la ampliación** de este proyecto; conservan su numeración para mantener la trazabilidad con las pruebas anteriores.

### Nivel 1: de síntoma a categoría de falla

```
R1: ∀n ( ErrorHTTP(n, 401) → CategoriaFalla(n, CredencialInvalida) )

R2: ∀n ( ErrorHTTP(n, 429) → CategoriaFalla(n, LimiteRateExcedido) )

R3: ∀n ( ∃c ( ErrorHTTP(n, c) ∧ c ≥ 500 ) → CategoriaFalla(n, ErrorServidorExterno) )
    equivalente a: ∀n ∀c ( ErrorHTTP(n, c) ∧ c ≥ 500 → CategoriaFalla(n, ErrorServidorExterno) )

R13: ∀n ( ErrorHTTP(n, 403) → CategoriaFalla(n, PermisosInsuficientes) )          [ampliación]
```

**Justificaciones**

- **R1:** el código 401 (*Unauthorized*) significa que el servicio no pudo autenticar la petición: la credencial es inválida, vencida o ausente.
- **R2:** 429 (*Too Many Requests*) es el código estándar que usan las APIs para indicar que se superó el límite de solicitudes.
- **R3:** los códigos 5xx indican fallas del lado del servidor, no de la configuración del workflow. El cuantificador ∃ en el antecedente equivale a un ∀ sobre el código `c`, que es la forma que se implementa: una variable `?c` con la restricción `?c >= 500`.
- **R13 (ampliación):** el código 403 (*Forbidden*) significa que el servicio sí reconoció la credencial pero niega el permiso solicitado; es distinto de un 401.

### Nivel 2: de categoría y contexto a causa raíz

```
R4: ∀n ( CategoriaFalla(n, CredencialInvalida) ∧ Autenticacion(n, OAuth2)
         → CausaRaiz(n, TokenVencido) )

R5: ∀n ( CategoriaFalla(n, CredencialInvalida) ∧ Autenticacion(n, ApiKey)
         → CausaRaiz(n, ApiKeyIncorrecta) )

R6: ∀n ( CategoriaFalla(n, LimiteRateExcedido) ∧ Contexto(n, ProcesaMuchosItems)
         → CausaRaiz(n, RafagaDeSolicitudes) )

R7: ∀n ( CategoriaFalla(n, ErrorServidorExterno) ∧ Sintoma(n, TimeoutNodo)
         → CausaRaiz(n, ServicioExternoSaturado) )

R14: ∀n ( CategoriaFalla(n, PermisosInsuficientes) ∧ Autenticacion(n, OAuth2)
         → CausaRaiz(n, ScopesInsuficientes) )                                  [ampliación]
```

**Justificaciones**

- **R4:** en OAuth2 los *access tokens* caducan (típicamente en 1 hora) y, si el refresh falla o el consentimiento fue revocado, el nodo recibe 401. Es la causa más frecuente de 401 con OAuth2.
- **R5:** una API Key no caduca por tiempo, así que un 401 indica que la clave fue copiada mal, regenerada o revocada.
- **R6:** n8n ejecuta el nodo una vez por item y sin pausa, por lo que con muchos items envía ráfagas que superan el límite por segundo o por minuto de la API.
- **R7:** un error 5xx acompañado de timeout indica que el servicio está sobrecargado o caído y no alcanza a responder.
- **R14 (ampliación):** en OAuth2 los permisos se otorgan mediante *scopes*; si el servicio responde 403, lo más probable es que la credencial se autorizó sin los scopes que el nodo necesita.

### Nivel 2 (continuación): causas directas desde síntomas

```
R8:  ∀w ( Sintoma(w, WebhookSinEjecuciones) ∧ Contexto(w, WorkflowInactivo)
          → CausaRaiz(w, WorkflowDesactivado) )

R9:  ∀w ( Sintoma(w, WebhookSinEjecuciones) ∧ Contexto(w, UsaURLTest)
          → CausaRaiz(w, URLDePruebaEnProduccion) )

R10: ∀n ( Sintoma(n, DatosVacios) ∧ Contexto(n, FiltroPrevioDescartaTodo)
          → CausaRaiz(n, FiltroDemasiadoRestrictivo) )

R11: ∀n ( Sintoma(n, ExpresionInvalida) → CausaRaiz(n, ReferenciaACampoInexistente) )

R15: ∀n ( Sintoma(n, TimeoutNodo) ∧ Contexto(n, ProcesaMuchosItems)
          → CausaRaiz(n, TimeoutPorVolumen) )                                    [ampliación]
```

**Justificaciones**

- **R8:** el webhook de producción solo se registra cuando el workflow está activo. Si está desactivado, las llamadas externas no disparan nada.
- **R9:** la URL `/webhook-test/` solo escucha mientras el editor está esperando un evento de prueba. Si el servicio externo la usa en operación normal, las llamadas se pierden.
- **R10:** en n8n un nodo sin items de entrada no se ejecuta. Si un IF o Filter previo descarta todo, los nodos siguientes aparecen "sin datos".
- **R11:** las expresiones del tipo `{{ $json.campo }}` fallan cuando el campo no existe, ya sea por un nombre mal escrito o por una estructura distinta de la esperada.
- **R15 (ampliación):** cuando un nodo que procesa un volumen alto de items excede el tiempo máximo, la causa más probable es el tamaño del lote y no la caída del servicio. A diferencia de R7, no exige un error 5xx.

### Nivel 3: de causa a recomendación

```
R12: ∀x ∀c ∀s ( CausaRaiz(x, c) ∧ Solucion(c, s) → Recomendar(x, s) )
```

**Justificación:** esta regla separa el **diagnóstico** (por qué falla) de la **acción** (qué hacer). Así se pueden cambiar las soluciones sin tocar las reglas de diagnóstico, y varias causas pueden compartir el mismo mecanismo de recomendación.

### Hechos estáticos: catálogo `Solucion(c, s)`

```
Solucion(TokenVencido,                 ReautenticarCredencialOAuth2)
Solucion(ApiKeyIncorrecta,             RegenerarYActualizarApiKey)
Solucion(RafagaDeSolicitudes,          UsarLoopOverItemsConWait)
Solucion(ServicioExternoSaturado,      ActivarRetryOnFailConEspera)
Solucion(WorkflowDesactivado,          ActivarElWorkflow)
Solucion(URLDePruebaEnProduccion,      UsarLaURLDeProduccionDelWebhook)
Solucion(FiltroDemasiadoRestrictivo,   RevisarCondicionesDelFiltro)
Solucion(ReferenciaACampoInexistente,  CorregirExpresionYVerificarEstructuraJSON)
Solucion(ScopesInsuficientes,          ReautorizarCredencialConScopesCorrectos)
Solucion(TimeoutPorVolumen,            DividirEnLotesConSplitInBatches)
```

## 4.4 Traducción a reglas de producción

### Conexión entre lógica de predicados y reglas de producción

Cada fórmula `∀x (A(x) ∧ B(x) → C(x))` se convierte en una regla de producción **SI A Y B ENTONCES C**. El cuantificador universal se implementa **instanciando la regla para cada objeto** (nodo o workflow) que aparece en los hechos, y el **Modus Ponens** (`A, A → B ⊢ B`) es el mecanismo que dispara la regla cuando todas sus condiciones se cumplen.

### Ejemplo de sintaxis SI-ENTONCES

```
R4: SI        CategoriaFalla(n, CredencialInvalida)
    Y         Autenticacion(n, OAuth2)
    ENTONCES  CausaRaiz(n, TokenVencido)   [CF = 0.80]
```

### Grupo 1: Reglas de síntoma → categoría

| Regla | SI | ENTONCES | CF |
|---|---|---|---|
| R1 | ErrorHTTP(n, 401) | CategoriaFalla(n, CredencialInvalida) | 0.90 |
| R2 | ErrorHTTP(n, 429) | CategoriaFalla(n, LimiteRateExcedido) | 0.95 |
| R3 | ErrorHTTP(n, c) **Y** c ≥ 500 | CategoriaFalla(n, ErrorServidorExterno) | 0.90 |
| R13 | ErrorHTTP(n, 403) | CategoriaFalla(n, PermisosInsuficientes) | 0.90 |

### Grupo 2: Reglas de categoría/contexto → causa raíz

| Regla | SI | ENTONCES | CF |
|---|---|---|---|
| R4 | CategoriaFalla(n, CredencialInvalida) **Y** Autenticacion(n, OAuth2) | CausaRaiz(n, TokenVencido) | 0.80 |
| R5 | CategoriaFalla(n, CredencialInvalida) **Y** Autenticacion(n, ApiKey) | CausaRaiz(n, ApiKeyIncorrecta) | 0.75 |
| R6 | CategoriaFalla(n, LimiteRateExcedido) **Y** Contexto(n, ProcesaMuchosItems) | CausaRaiz(n, RafagaDeSolicitudes) | 0.90 |
| R7 | CategoriaFalla(n, ErrorServidorExterno) **Y** Sintoma(n, TimeoutNodo) | CausaRaiz(n, ServicioExternoSaturado) | 0.75 |
| R14 | CategoriaFalla(n, PermisosInsuficientes) **Y** Autenticacion(n, OAuth2) | CausaRaiz(n, ScopesInsuficientes) | 0.80 |

### Grupo 3: Reglas de síntoma → causa raíz (activación, datos y expresiones)

| Regla | SI | ENTONCES | CF |
|---|---|---|---|
| R8 | Sintoma(w, WebhookSinEjecuciones) **Y** Contexto(w, WorkflowInactivo) | CausaRaiz(w, WorkflowDesactivado) | 0.95 |
| R9 | Sintoma(w, WebhookSinEjecuciones) **Y** Contexto(w, UsaURLTest) | CausaRaiz(w, URLDePruebaEnProduccion) | 0.90 |
| R10 | Sintoma(n, DatosVacios) **Y** Contexto(n, FiltroPrevioDescartaTodo) | CausaRaiz(n, FiltroDemasiadoRestrictivo) | 0.85 |
| R11 | Sintoma(n, ExpresionInvalida) | CausaRaiz(n, ReferenciaACampoInexistente) | 0.80 |
| R15 | Sintoma(n, TimeoutNodo) **Y** Contexto(n, ProcesaMuchosItems) | CausaRaiz(n, TimeoutPorVolumen) | 0.80 |

### Grupo 4: Regla de recomendación

| Regla | SI | ENTONCES | CF |
|---|---|---|---|
| R12 | CausaRaiz(x, c) **Y** Solucion(c, s) | Recomendar(x, s) | 1.00 |

### Cómo se escribe una regla en el código

Una regla es un diccionario con `nombre`, `descripcion`, `grupo`, `condiciones`, `restricciones`, `conclusion` y `certeza`. Un texto que empieza con `?` es una **variable**, que corresponde al cuantificador ∀:

```python
# R3: ∀n ∀c ( ErrorHTTP(n, c) ∧ c ≥ 500 → CategoriaFalla(n, ErrorServidorExterno) )
crear_regla("R3", "Un código ≥ 500 indica error del servidor externo", "Síntoma → categoría",
            [("ErrorHTTP", "?c")], ("CategoriaFalla", "ErrorServidorExterno"), 0.90,
            restricciones=[("?c", ">=", 500)])

# R12: ∀c ∀s ( CausaRaiz(c) ∧ Solucion(c, s) → Recomendar(s) )
crear_regla("R12", "...", "Recomendación",
            [("CausaRaiz", "?c"), ("Solucion", "?c", "?s")], ("Recomendar", "?s"), 1.00)
```

El motor busca todas las formas de ligar las variables para que **todas** las condiciones coincidan con hechos conocidos (Modus Ponens) y comprueba las restricciones antes de disparar la regla.

## 4.5 Factores de certeza

### Decisión de diseño

**Sí se usan factores de certeza (CF).** El diagnóstico de fallas no es determinista: un 401 con OAuth2 es *casi siempre* un token vencido, pero también puede ser un *scope* incorrecto o una cuenta suspendida. Por eso cada regla tiene un CF entre 0 y 1 que expresa qué tan fiable es esa inferencia según la experiencia del experto.

### Esquema de propagación

| Situación | Fórmula |
|---|---|
| Hechos observados | CF = 1.0 (salvo que se indique otro valor) |
| Conclusión de una regla | `CF(conclusión) = min(CF de las condiciones) × CF(regla)` |
| Misma conclusión por dos caminos | El motor **conserva el CF de la primera derivación** y lo registra en el log. Ninguna regla de la base actual produce esta situación |

### Ejemplo de cálculo (401 con OAuth2)

```
Hechos:  ErrorHTTP(n2, 401) [1.0],  Autenticacion(n2, OAuth2) [1.0]

R1:  CategoriaFalla(n2, CredencialInvalida)        = min(1.0)        × 0.90 = 0.90
R4:  CausaRaiz(n2, TokenVencido)                   = min(0.90, 1.0)  × 0.80 = 0.72
R12: Recomendar(n2, ReautenticarCredencialOAuth2)  = min(0.72, 1.0)  × 1.00 = 0.72
```

---
# 5. Implementación

El código está organizado **por módulos**, cada uno en su propia celda. Cuando se pase a la estructura de carpetas (sección 8), cada módulo se copia tal cual a su archivo de `src/`. La lógica **no imprime nada**: devuelve datos y texto, y es el notebook (o la interfaz web) quien los muestra.

| Módulo (celda) | Contenido |
|---|---|
| `base_conocimiento` | `DOMINIOS`, `PREDICADOS`, `SOLUCIONES`, `crear_regla()`, `REGLAS`, `BASE_CONOCIMIENTO` |
| `motor_inferencia` | `encadenamiento_adelante()`, `explicar_razonamiento()`, `porque()`, `validar_hechos()`, `detectar_ciclos()`, `diagnosticar()` |
| `casos_prueba` | `CASOS_PRUEBA` y `ejecutar_caso()` (sección 6) |

## 5.1 Base de conocimiento (`base_conocimiento.py`)

In [ ]:
# ============================================================
# MÓDULO: base_conocimiento.py
# Predicados, dominios, reglas de producción y catálogo de soluciones
# ============================================================

# --- Dominios (universos de discurso) usados para validar la entrada ---
DOMINIOS = {
    "K": range(100, 600),                                   # códigos HTTP
    "T": ["OAuth2", "ApiKey"],                              # tipos de autenticación
    "SIN": ["WebhookSinEjecuciones", "DatosVacios", "ExpresionInvalida", "TimeoutNodo"],
    "CTX": ["ProcesaMuchosItems", "WorkflowInactivo", "UsaURLTest", "FiltroPrevioDescartaTodo"],
    "F": ["CredencialInvalida", "LimiteRateExcedido", "ErrorServidorExterno", "PermisosInsuficientes"],
    "C": ["TokenVencido", "ApiKeyIncorrecta", "RafagaDeSolicitudes", "ServicioExternoSaturado",
          "WorkflowDesactivado", "URLDePruebaEnProduccion", "FiltroDemasiadoRestrictivo",
          "ReferenciaACampoInexistente", "ScopesInsuficientes", "TimeoutPorVolumen"],
    "S": ["ReautenticarCredencialOAuth2", "RegenerarYActualizarApiKey", "UsarLoopOverItemsConWait",
          "ActivarRetryOnFailConEspera", "ActivarElWorkflow", "UsarLaURLDeProduccionDelWebhook",
          "RevisarCondicionesDelFiltro", "CorregirExpresionYVerificarEstructuraJSON",
          "ReautorizarCredencialConScopesCorrectos", "DividirEnLotesConSplitInBatches"],
}

# --- Predicados: 8 en total. "argumentos" = dominios de los argumentos que aparecen
#     en los hechos (el objeto n/x diagnosticado se omite: se diagnostica uno a la vez).
PREDICADOS = {
    "ErrorHTTP":      {"formal": "ErrorHTTP(n, c)",      "argumentos": ["K"],        "capa": "observacion",
                       "descripcion": "El nodo n respondió con el código de error HTTP c"},
    "Autenticacion":  {"formal": "Autenticacion(n, t)",  "argumentos": ["T"],        "capa": "observacion",
                       "descripcion": "El nodo n se autentica mediante el mecanismo t"},
    "Sintoma":        {"formal": "Sintoma(x, s)",        "argumentos": ["SIN"],      "capa": "observacion",
                       "descripcion": "El objeto x presenta el síntoma s"},
    "Contexto":       {"formal": "Contexto(x, k)",       "argumentos": ["CTX"],      "capa": "observacion",
                       "descripcion": "El objeto x cumple la condición de contexto k"},
    "CategoriaFalla": {"formal": "CategoriaFalla(n, f)", "argumentos": ["F"],        "capa": "intermedio",
                       "descripcion": "La falla del nodo n pertenece a la categoría general f"},
    "CausaRaiz":      {"formal": "CausaRaiz(x, c)",      "argumentos": ["C"],        "capa": "diagnostico",
                       "descripcion": "La causa probable de la falla del objeto x es c"},
    "Solucion":       {"formal": "Solucion(c, s)",       "argumentos": ["C", "S"],   "capa": "catalogo",
                       "descripcion": "La acción s corrige la causa c (catálogo fijo)"},
    "Recomendar":     {"formal": "Recomendar(x, s)",     "argumentos": ["S"],        "capa": "diagnostico",
                       "descripcion": "Se recomienda aplicar la acción s sobre el objeto x"},
}

# --- Catálogo Solucion(c, s) ---
SOLUCIONES = {
    "TokenVencido":                "ReautenticarCredencialOAuth2",
    "ApiKeyIncorrecta":            "RegenerarYActualizarApiKey",
    "RafagaDeSolicitudes":         "UsarLoopOverItemsConWait",
    "ServicioExternoSaturado":     "ActivarRetryOnFailConEspera",
    "WorkflowDesactivado":         "ActivarElWorkflow",
    "URLDePruebaEnProduccion":     "UsarLaURLDeProduccionDelWebhook",
    "FiltroDemasiadoRestrictivo":  "RevisarCondicionesDelFiltro",
    "ReferenciaACampoInexistente": "CorregirExpresionYVerificarEstructuraJSON",
    "ScopesInsuficientes":         "ReautorizarCredencialConScopesCorrectos",
    "TimeoutPorVolumen":           "DividirEnLotesConSplitInBatches",
}


def crear_regla(nombre, descripcion, grupo, condiciones, conclusion, certeza=1.0, restricciones=None):
    """
    Crea una regla de producción.

    condiciones:   lista de patrones (predicado, arg, ...). Un texto que empieza con "?"
                   es una variable (equivale al cuantificador ∀ de la lógica de predicados).
    restricciones: lista de (variable, operador, valor), p. ej. ("?c", ">=", 500).
    """
    return {
        "nombre": nombre, "descripcion": descripcion, "grupo": grupo,
        "condiciones": list(condiciones), "restricciones": list(restricciones or []),
        "conclusion": conclusion, "certeza": certeza,
    }


G1, G2, G3, G4 = ("Síntoma → categoría", "Categoría/contexto → causa raíz",
                  "Síntoma → causa raíz", "Recomendación")

REGLAS = [
    crear_regla("R1", "Un 401 indica credencial inválida", G1,
                [("ErrorHTTP", 401)], ("CategoriaFalla", "CredencialInvalida"), 0.90),
    crear_regla("R2", "Un 429 indica límite de solicitudes excedido", G1,
                [("ErrorHTTP", 429)], ("CategoriaFalla", "LimiteRateExcedido"), 0.95),
    crear_regla("R3", "Un código ≥ 500 indica error del servidor externo", G1,
                [("ErrorHTTP", "?c")], ("CategoriaFalla", "ErrorServidorExterno"), 0.90,
                restricciones=[("?c", ">=", 500)]),
    crear_regla("R4", "Credencial inválida con OAuth2: token vencido", G2,
                [("CategoriaFalla", "CredencialInvalida"), ("Autenticacion", "OAuth2")],
                ("CausaRaiz", "TokenVencido"), 0.80),
    crear_regla("R5", "Credencial inválida con API Key: clave incorrecta", G2,
                [("CategoriaFalla", "CredencialInvalida"), ("Autenticacion", "ApiKey")],
                ("CausaRaiz", "ApiKeyIncorrecta"), 0.75),
    crear_regla("R6", "Límite excedido con muchos items: ráfaga de solicitudes", G2,
                [("CategoriaFalla", "LimiteRateExcedido"), ("Contexto", "ProcesaMuchosItems")],
                ("CausaRaiz", "RafagaDeSolicitudes"), 0.90),
    crear_regla("R7", "Error del servidor con timeout: servicio saturado", G2,
                [("CategoriaFalla", "ErrorServidorExterno"), ("Sintoma", "TimeoutNodo")],
                ("CausaRaiz", "ServicioExternoSaturado"), 0.75),
    crear_regla("R8", "Webhook sin ejecuciones con workflow inactivo", G3,
                [("Sintoma", "WebhookSinEjecuciones"), ("Contexto", "WorkflowInactivo")],
                ("CausaRaiz", "WorkflowDesactivado"), 0.95),
    crear_regla("R9", "Webhook sin ejecuciones con URL de prueba en uso", G3,
                [("Sintoma", "WebhookSinEjecuciones"), ("Contexto", "UsaURLTest")],
                ("CausaRaiz", "URLDePruebaEnProduccion"), 0.90),
    crear_regla("R10", "Datos vacíos tras un filtro que descarta todo", G3,
                [("Sintoma", "DatosVacios"), ("Contexto", "FiltroPrevioDescartaTodo")],
                ("CausaRaiz", "FiltroDemasiadoRestrictivo"), 0.85),
    crear_regla("R11", "Expresión inválida: referencia a un campo inexistente", G3,
                [("Sintoma", "ExpresionInvalida")],
                ("CausaRaiz", "ReferenciaACampoInexistente"), 0.80),
    crear_regla("R12", "Toda causa raíz con solución en el catálogo genera una recomendación", G4,
                [("CausaRaiz", "?c"), ("Solucion", "?c", "?s")],
                ("Recomendar", "?s"), 1.00),
    # --- Ampliación respecto a la Actividad 4 ---
    crear_regla("R13", "Un 403 indica permisos insuficientes", G1,
                [("ErrorHTTP", 403)], ("CategoriaFalla", "PermisosInsuficientes"), 0.90),
    crear_regla("R14", "Permisos insuficientes con OAuth2: scopes insuficientes", G2,
                [("CategoriaFalla", "PermisosInsuficientes"), ("Autenticacion", "OAuth2")],
                ("CausaRaiz", "ScopesInsuficientes"), 0.80),
    crear_regla("R15", "Timeout con muchos items: lote demasiado grande", G3,
                [("Sintoma", "TimeoutNodo"), ("Contexto", "ProcesaMuchosItems")],
                ("CausaRaiz", "TimeoutPorVolumen"), 0.80),
]

# Hechos estáticos: el catálogo Solucion(c, s) entra al motor como hechos con CF = 1.0
HECHOS_ESTATICOS = [("Solucion", c, s) for c, s in SOLUCIONES.items()]


def crear_base_conocimiento():
    """Retorna la base de conocimiento lista para el motor de inferencia."""
    return {"reglas": REGLAS, "hechos_estaticos": HECHOS_ESTATICOS}


BASE_CONOCIMIENTO = crear_base_conocimiento()


## 5.2 Motor de inferencia (`motor_inferencia.py`)

### Funciones del motor

| Función | Responsabilidad |
|---|---|
| `encadenamiento_adelante(base, hechos)` | Aplica las reglas hasta el punto fijo. Retorna `(hechos_finales, reglas_aplicadas)` |
| `explicar_razonamiento(reglas_aplicadas)` | Retorna el texto con todos los pasos de la inferencia |
| `porque(conclusion, hechos_iniciales, reglas_aplicadas)` | Retorna la cadena que justifica una conclusión, hasta los hechos observados |
| `validar_hechos(hechos)` | Valida la entrada. Retorna `(errores, advertencias)` |
| `detectar_ciclos(base)` | Busca ciclos entre predicados en la base de reglas |
| `diagnosticar(hechos)` | Orquesta todo y retorna un diccionario listo para la interfaz |

### Algoritmo de encadenamiento hacia adelante

1. Se cargan los hechos estáticos (catálogo `Solucion`) y los hechos iniciales, todos con CF = 1.0. Los duplicados se ignoran.
2. En cada iteración se recorren las reglas. Para cada una se buscan **todas las formas de ligar sus variables** de modo que todas las condiciones coincidan con hechos conocidos y se cumplan las restricciones.
3. Si la conclusión es nueva, se agrega a los hechos con `CF = min(CF de las condiciones) × CF de la regla` y se registra la regla aplicada.
4. Se repite hasta que una iteración completa no produce hechos nuevos (**punto fijo**).

### Casos especiales

| Situación | Cómo se maneja |
|---|---|
| Sin conclusión | `estado = "sin_conclusion"` con un mensaje claro; si solo se deriva información intermedia, `"diagnostico_parcial"` |
| Entrada inválida | `validar_hechos()` devuelve errores y `diagnosticar()` responde `"entrada_invalida"` sin ejecutar el motor |
| Ciclos infinitos | Tres protecciones: `detectar_ciclos()` avisa de ciclos entre predicados; una regla no se dispara dos veces con los mismos hechos; y `max_iteraciones` detiene el motor |
| Registro del proceso | Módulo `logging` (logger `sbc.motor`): INFO por cada regla disparada, DEBUG para conclusiones repetidas, WARNING ante ciclos o límite excedido |


In [2]:
# ============================================================
# MÓDULO: motor_inferencia.py
# Encadenamiento hacia adelante con factores de certeza y explicación
# ============================================================
import logging

# En src/motor_inferencia.py descomentar la siguiente línea (en el notebook las celdas comparten espacio de nombres):
# from base_conocimiento import PREDICADOS, DOMINIOS, BASE_CONOCIMIENTO

logger = logging.getLogger("sbc.motor")

OPERADORES = {
    ">=": lambda a, b: a >= b, "<=": lambda a, b: a <= b,
    ">":  lambda a, b: a > b,  "<":  lambda a, b: a < b,
    "==": lambda a, b: a == b, "!=": lambda a, b: a != b,
}


# ---------- Utilidades de representación ----------
def formatear_hecho(hecho):
    """('ErrorHTTP', 401) -> 'ErrorHTTP(401)'"""
    nombre, *args = hecho
    return f"{nombre}({', '.join(str(a) for a in args)})"


def calcular_certeza(certeza_condiciones, certeza_regla):
    """CF(conclusión) = min(CF de las condiciones) × CF(regla), redondeado a 2 decimales."""
    return round(certeza_condiciones * certeza_regla, 2)


# ---------- Unificación de patrones (implementa el ∀ de las reglas) ----------
def _es_variable(x):
    return isinstance(x, str) and x.startswith("?")


def _unificar(patron, hecho, ligas):
    """Intenta ajustar un patrón a un hecho; retorna las ligas ampliadas o None."""
    if len(patron) != len(hecho):
        return None
    nuevas = dict(ligas)
    for p, h in zip(patron, hecho):
        if _es_variable(p):
            if p in nuevas and nuevas[p] != h:
                return None
            nuevas[p] = h
        elif p != h:
            return None
    return nuevas


def _buscar_ligas(condiciones, hechos, ligas=None):
    """Genera (ligas, hechos_usados) para cada forma de satisfacer TODAS las condiciones."""
    ligas = ligas or {}
    if not condiciones:
        yield ligas, []
        return
    primera, *resto = condiciones
    for hecho in hechos:
        nuevas = _unificar(primera, hecho, ligas)
        if nuevas is not None:
            for l, usados in _buscar_ligas(resto, hechos, nuevas):
                yield l, [hecho] + usados


def _cumple_restricciones(restricciones, ligas):
    for variable, operador, valor in restricciones:
        try:
            if not OPERADORES[operador](ligas[variable], valor):
                return False
        except TypeError:          # p. ej. comparar un texto con un número
            return False
    return True


def _instanciar(patron, ligas):
    return tuple(ligas.get(x, x) if _es_variable(x) else x for x in patron)


# ---------- Detección de ciclos en la base de reglas ----------
def detectar_ciclos(base_conocimiento):
    """
    Busca ciclos entre predicados (A → B → A) en la base de reglas.
    Retorna una lista de ciclos, cada uno como lista de nombres de predicado.
    """
    grafo = {}
    for regla in base_conocimiento["reglas"]:
        destino = regla["conclusion"][0]
        for condicion in regla["condiciones"]:
            grafo.setdefault(condicion[0], set()).add(destino)

    ciclos, estado, pila = [], {}, []

    def visitar(nodo):
        estado[nodo] = 1
        pila.append(nodo)
        for vecino in sorted(grafo.get(nodo, ())):
            if estado.get(vecino, 0) == 0:
                visitar(vecino)
            elif estado[vecino] == 1:
                ciclos.append(pila[pila.index(vecino):] + [vecino])
        pila.pop()
        estado[nodo] = 2

    for nodo in sorted(grafo):
        if estado.get(nodo, 0) == 0:
            visitar(nodo)
    return ciclos


# ---------- Motor de inferencia ----------
def _ejecutar(base_conocimiento, hechos_iniciales, max_iteraciones=100):
    hechos, certezas, aplicadas, disparos = [], {}, [], set()

    for h in list(base_conocimiento.get("hechos_estaticos", [])) + [tuple(h) for h in hechos_iniciales]:
        if h not in certezas:                       # hechos duplicados se ignoran
            hechos.append(h)
            certezas[h] = 1.0

    ciclos_reglas = detectar_ciclos(base_conocimiento)
    if ciclos_reglas:
        logger.warning("La base de reglas contiene ciclos entre predicados: %s", ciclos_reglas)
    logger.info("Inicio de inferencia con %d hechos iniciales", len(hechos_iniciales))

    iteraciones, limite_excedido, cambio = 0, False, True
    while cambio:
        iteraciones += 1
        if iteraciones > max_iteraciones:           # protección contra ciclos infinitos
            limite_excedido = True
            logger.warning("Se alcanzó el límite de %d iteraciones; inferencia detenida", max_iteraciones)
            break
        cambio = False
        for regla in base_conocimiento["reglas"]:
            for ligas, usados in list(_buscar_ligas(regla["condiciones"], list(hechos))):
                if not _cumple_restricciones(regla["restricciones"], ligas):
                    continue
                firma = (regla["nombre"], tuple(sorted(ligas.items())))
                if firma in disparos:               # una regla no se dispara dos veces con lo mismo
                    continue
                disparos.add(firma)
                conclusion = _instanciar(regla["conclusion"], ligas)
                if conclusion in certezas:
                    logger.debug("%s: %s ya estaba derivada; se conserva su CF", regla["nombre"],
                                 formatear_hecho(conclusion))
                    continue
                cf = calcular_certeza(min([certezas[u] for u in usados] or [1.0]), regla["certeza"])
                hechos.append(conclusion)
                certezas[conclusion] = cf
                aplicadas.append({"regla": regla["nombre"], "descripcion": regla["descripcion"],
                                  "condiciones": usados, "conclusion": conclusion, "certeza": cf})
                logger.info("Iteración %d | %s: %s ⇒ %s [CF %.2f]", iteraciones, regla["nombre"],
                            " ∧ ".join(formatear_hecho(u) for u in usados), formatear_hecho(conclusion), cf)
                cambio = True

    logger.info("Fin de inferencia: %d reglas aplicadas en %d iteraciones", len(aplicadas), iteraciones)
    return {"hechos": hechos, "certezas": certezas, "reglas_aplicadas": aplicadas,
            "iteraciones": iteraciones, "limite_excedido": limite_excedido, "ciclos_reglas": ciclos_reglas}


def encadenamiento_adelante(base_conocimiento, hechos_iniciales, max_iteraciones=100):
    """
    Aplica encadenamiento hacia adelante hasta el punto fijo.
    Returns: (hechos_finales, reglas_aplicadas)
    """
    r = _ejecutar(base_conocimiento, hechos_iniciales, max_iteraciones)
    return r["hechos"], r["reglas_aplicadas"]


# ---------- Explicación ----------
def explicar_razonamiento(reglas_aplicadas):
    """Genera la explicación legible de todo el razonamiento. Returns: texto."""
    if not reglas_aplicadas:
        return ("No se aplicó ninguna regla: con los hechos ingresados no es posible "
                "derivar conclusiones.")
    lineas = ["EXPLICACIÓN DEL RAZONAMIENTO"]
    for i, r in enumerate(reglas_aplicadas, start=1):
        lineas += [f"\nPaso {i} · {r['regla']}: {r['descripcion']}",
                   "   SI        " + "  Y  ".join(formatear_hecho(c) for c in r["condiciones"]),
                   f"   ENTONCES  {formatear_hecho(r['conclusion'])}   [CF = {r['certeza']:.2f}]"]
    return "\n".join(lineas)


def porque(conclusion, hechos_iniciales, reglas_aplicadas):
    """Explica por qué se llegó a una conclusión, recorriendo la cadena hasta los hechos observados."""
    conclusion = tuple(conclusion)
    iniciales = {tuple(h) for h in hechos_iniciales}
    por_conclusion = {r["conclusion"]: r for r in reglas_aplicadas}

    if conclusion not in por_conclusion:
        if conclusion in iniciales:
            return f"{formatear_hecho(conclusion)} es un hecho observado (dato de entrada); no fue derivado."
        return f"No se derivó {formatear_hecho(conclusion)} con los hechos ingresados."

    lineas = []

    def expandir(hecho, nivel, visitados):
        sangria = "    " * nivel
        if hecho in por_conclusion and hecho not in visitados:
            r = por_conclusion[hecho]
            lineas.append(f"{sangria}• {formatear_hecho(hecho)}  → derivado por {r['regla']} [CF {r['certeza']:.2f}]")
            for cond in r["condiciones"]:
                expandir(cond, nivel + 1, visitados | {hecho})
        elif hecho in iniciales:
            lineas.append(f"{sangria}• {formatear_hecho(hecho)}  → hecho observado")
        else:
            lineas.append(f"{sangria}• {formatear_hecho(hecho)}  → conocimiento de la base (catálogo)")

    r = por_conclusion[conclusion]
    lineas.append(f"¿Por qué {formatear_hecho(conclusion)}?  (se aplicó {r['regla']}: {r['descripcion']})")
    for cond in r["condiciones"]:
        expandir(cond, 1, {conclusion})
    return "\n".join(lineas)


# ---------- Validación de entrada ----------
def validar_hechos(hechos, predicados=None, dominios=None):
    """
    Validación básica de los hechos de entrada.
    Returns: (errores, advertencias)
    """
    predicados = PREDICADOS if predicados is None else predicados
    dominios = DOMINIOS if dominios is None else dominios
    errores, advertencias, vistos = [], [], set()

    if not hechos:
        advertencias.append("No se ingresó ningún hecho: el sistema no podrá derivar conclusiones.")
    for h in hechos:
        h = tuple(h)
        if not h:
            errores.append("Se recibió un hecho vacío.")
            continue
        nombre, *args = h
        if nombre not in predicados:
            errores.append(f"Predicado desconocido: «{nombre}».")
            continue
        info = predicados[nombre]
        if info["capa"] != "observacion":
            errores.append(f"«{nombre}» no es un dato observable: lo deriva el sistema.")
            continue
        if len(args) != len(info["argumentos"]):
            errores.append(f"{nombre} espera {len(info['argumentos'])} argumento(s) y recibió {len(args)}.")
            continue
        for arg, dom in zip(args, info["argumentos"]):
            valido = (isinstance(arg, int) and not isinstance(arg, bool) and arg in dominios[dom]) \
                if dom == "K" else arg in dominios[dom]
            if not valido:
                errores.append(f"Valor no válido para {nombre}: {arg!r} (dominio {dom}).")
        if h in vistos:
            advertencias.append(f"Hecho repetido: {formatear_hecho(h)} (se considera una sola vez).")
        vistos.add(h)

    if ("Autenticacion", "OAuth2") in vistos and ("Autenticacion", "ApiKey") in vistos:
        advertencias.append("Se indicaron OAuth2 y ApiKey a la vez; un mismo nodo usa un solo mecanismo.")
    return errores, advertencias


# ---------- Diagnóstico completo (lo que consumirá la interfaz web) ----------
def diagnosticar(hechos_iniciales, base_conocimiento=None, max_iteraciones=100, validar=True):
    """
    Ejecuta el diagnóstico completo. No imprime nada: retorna un diccionario
    que la interfaz web (Streamlit) puede mostrar directamente.
    """
    base = BASE_CONOCIMIENTO if base_conocimiento is None else base_conocimiento
    hechos_iniciales = [tuple(h) for h in hechos_iniciales]
    errores, advertencias = validar_hechos(hechos_iniciales) if validar else ([], [])

    resultado = {"hechos_iniciales": hechos_iniciales, "errores": errores, "advertencias": advertencias,
                 "hechos_finales": [], "hechos_derivados": [], "reglas_aplicadas": [], "causas": [],
                 "iteraciones": 0, "limite_excedido": False, "ciclos_reglas": [],
                 "explicacion": "", "estado": "entrada_invalida", "mensaje": ""}
    if errores:
        resultado["mensaje"] = "La entrada contiene errores: " + " ".join(errores)
        return resultado

    r = _ejecutar(base, hechos_iniciales, max_iteraciones)
    aplicadas, certezas = r["reglas_aplicadas"], r["certezas"]
    derivados = [a["conclusion"] for a in aplicadas]

    causas = []
    for h in derivados:
        if h[0] == "CausaRaiz":
            rec = next((a["conclusion"][1] for a in aplicadas
                        if a["conclusion"][0] == "Recomendar" and h in a["condiciones"]), None)
            causas.append({"causa": h[1], "certeza": certezas[h], "recomendacion": rec})

    if causas and all(c["recomendacion"] for c in causas):
        estado, mensaje = "diagnostico_completo", f"Se identificaron {len(causas)} causa(s) probable(s) con su acción recomendada."
    elif aplicadas:
        estado = "diagnostico_parcial"
        mensaje = ("Se pudo derivar información parcial (" + ", ".join(formatear_hecho(d) for d in derivados) +
                   "), pero no hay datos suficientes para determinar la causa raíz. "
                   "Agregue el tipo de autenticación, síntomas o factores de contexto.")
    else:
        estado = "sin_conclusion"
        mensaje = ("Con los hechos ingresados no se pudo derivar ninguna conclusión. Puede que el caso esté "
                   "fuera del alcance del sistema (p. ej. códigos 400 o 404) o que falten datos.")

    resultado.update({"hechos_finales": r["hechos"], "hechos_derivados": derivados, "reglas_aplicadas": aplicadas,
                      "causas": causas, "iteraciones": r["iteraciones"], "limite_excedido": r["limite_excedido"],
                      "ciclos_reglas": r["ciclos_reglas"], "explicacion": explicar_razonamiento(aplicadas),
                      "estado": estado, "mensaje": mensaje})
    return resultado


## 5.3 Demostración del motor

Diagnóstico de un 401 con OAuth2, con el log del proceso de inferencia activado (aparece en rojo, es la salida de error del notebook):

In [3]:
logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(message)s")
logging.getLogger("sbc").setLevel(logging.INFO)          # activar el log de la inferencia

hechos = [("ErrorHTTP", 401), ("Autenticacion", "OAuth2")]
resultado = diagnosticar(hechos)

logging.getLogger("sbc").setLevel(logging.WARNING)       # volver al nivel normal

print("ESTADO:", resultado["estado"])
print(resultado["mensaje"])
for c in resultado["causas"]:
    print(f"  Causa raíz: {c['causa']}  |  Recomendación: {c['recomendacion']}  |  CF = {c['certeza']}")
print()
print(resultado["explicacion"])

INFO:sbc.motor:Inicio de inferencia con 2 hechos iniciales
INFO:sbc.motor:Iteración 1 | R1: ErrorHTTP(401) ⇒ CategoriaFalla(CredencialInvalida) [CF 0.90]
INFO:sbc.motor:Iteración 1 | R4: CategoriaFalla(CredencialInvalida) ∧ Autenticacion(OAuth2) ⇒ CausaRaiz(TokenVencido) [CF 0.72]
INFO:sbc.motor:Iteración 1 | R12: CausaRaiz(TokenVencido) ∧ Solucion(TokenVencido, ReautenticarCredencialOAuth2) ⇒ Recomendar(ReautenticarCredencialOAuth2) [CF 0.72]
INFO:sbc.motor:Fin de inferencia: 3 reglas aplicadas en 2 iteraciones


ESTADO: diagnostico_completo
Se identificaron 1 causa(s) probable(s) con su acción recomendada.
  Causa raíz: TokenVencido  |  Recomendación: ReautenticarCredencialOAuth2  |  CF = 0.72

EXPLICACIÓN DEL RAZONAMIENTO

Paso 1 · R1: Un 401 indica credencial inválida
   SI        ErrorHTTP(401)
   ENTONCES  CategoriaFalla(CredencialInvalida)   [CF = 0.90]

Paso 2 · R4: Credencial inválida con OAuth2: token vencido
   SI        CategoriaFalla(CredencialInvalida)  Y  Autenticacion(OAuth2)
   ENTONCES  CausaRaiz(TokenVencido)   [CF = 0.72]

Paso 3 · R12: Toda causa raíz con solución en el catálogo genera una recomendación
   SI        CausaRaiz(TokenVencido)  Y  Solucion(TokenVencido, ReautenticarCredencialOAuth2)
   ENTONCES  Recomendar(ReautenticarCredencialOAuth2)   [CF = 0.72]


In [4]:
# ¿Por qué se llegó a la recomendación? La cadena llega hasta los hechos observados.
print(porque(("Recomendar", "ReautenticarCredencialOAuth2"), resultado["hechos_iniciales"], resultado["reglas_aplicadas"]))

¿Por qué Recomendar(ReautenticarCredencialOAuth2)?  (se aplicó R12: Toda causa raíz con solución en el catálogo genera una recomendación)
    • CausaRaiz(TokenVencido)  → derivado por R4 [CF 0.72]
        • CategoriaFalla(CredencialInvalida)  → derivado por R1 [CF 0.90]
            • ErrorHTTP(401)  → hecho observado
        • Autenticacion(OAuth2)  → hecho observado
    • Solucion(TokenVencido, ReautenticarCredencialOAuth2)  → conocimiento de la base (catálogo)


Casos que el sistema debe manejar sin fallar: una entrada sin conclusión y una entrada inválida.

In [5]:
print("404 →", diagnosticar([("ErrorHTTP", 404)])["mensaje"])
print()
print("999 →", diagnosticar([("ErrorHTTP", 999)])["mensaje"])

404 → Con los hechos ingresados no se pudo derivar ninguna conclusión. Puede que el caso esté fuera del alcance del sistema (p. ej. códigos 400 o 404) o que falten datos.

999 → La entrada contiene errores: Valor no válido para ErrorHTTP: 999 (dominio K).


---
# 6. Casos de prueba y validación

Los casos están definidos como **datos** (hechos, reglas esperadas, causas esperadas con su CF y estado esperado) y se ejecutan automáticamente. La validación compara lo esperado con lo obtenido, de modo que la columna «Estado» se calcula y no se escribe a mano.

## 6.1 Diseño de los casos

| Tipo | Casos | Qué ejercita |
|---|---|---|
| Funcional | 1 a 10 | Una ruta de razonamiento distinta por caso. Los casos 1 a 6 vienen de la Actividad 4; el 7 y el 8 cubren R5 y R11, que antes no se probaban; el 9 y el 10 cubren la ampliación (R13 a R15) |
| Multifalla | 11 | Un workflow con fallas simultáneas: activa **14 de las 15 reglas** y deriva 9 causas a la vez |
| Sin conclusión | 12, 13 | Entrada vacía y un código fuera del alcance (404) |
| Borde | 14 a 17 | Diagnóstico parcial (401 sin autenticación, 503 sin timeout), hechos duplicados y una base con un ciclo |

> **Nota sobre el caso 11:** los hechos no llevan el nodo al que pertenecen, así que representa un workflow con varias fallas simultáneas, no varios nodos distintos (ver limitaciones en la sección 7).

In [6]:
# ============================================================
# MÓDULO: casos_prueba.py  — datos de los casos de prueba
# En src/casos_prueba.py agregar:
#   from base_conocimiento import crear_regla
#   from motor_inferencia import diagnosticar, formatear_hecho
# ============================================================
TOY_CICLO = {   # base de juguete con un ciclo A → B → A, para probar la protección del motor
    "reglas": [crear_regla("RA", "A implica B", "prueba", [("A",)], ("B",)),
               crear_regla("RB", "B implica A", "prueba", [("B",)], ("A",))],
    "hechos_estaticos": [],
}

TODAS_LAS_CAUSAS_MULTIFALLA = {
    "TokenVencido": 0.72, "RafagaDeSolicitudes": 0.85, "ServicioExternoSaturado": 0.68,
    "WorkflowDesactivado": 0.95, "URLDePruebaEnProduccion": 0.90, "FiltroDemasiadoRestrictivo": 0.85,
    "ReferenciaACampoInexistente": 0.80, "ScopesInsuficientes": 0.72, "TimeoutPorVolumen": 0.80,
}

CASOS_PRUEBA = [
    # ---------- Casos funcionales (cada uno ejercita una ruta distinta) ----------
    dict(id=1, tipo="Funcional", descripcion="401 con OAuth2",
         hechos=[("ErrorHTTP", 401), ("Autenticacion", "OAuth2")],
         reglas=["R1", "R4", "R12"], causas={"TokenVencido": 0.72}, estado="diagnostico_completo",
         conclusion="TokenVencido → ReautenticarCredencialOAuth2"),
    dict(id=2, tipo="Funcional", descripcion="429 con muchos items",
         hechos=[("ErrorHTTP", 429), ("Contexto", "ProcesaMuchosItems")],
         reglas=["R2", "R6", "R12"], causas={"RafagaDeSolicitudes": 0.85}, estado="diagnostico_completo",
         conclusion="RafagaDeSolicitudes → UsarLoopOverItemsConWait"),
    dict(id=3, tipo="Funcional", descripcion="503 con timeout",
         hechos=[("ErrorHTTP", 503), ("Sintoma", "TimeoutNodo")],
         reglas=["R3", "R7", "R12"], causas={"ServicioExternoSaturado": 0.68}, estado="diagnostico_completo",
         conclusion="ServicioExternoSaturado → ActivarRetryOnFailConEspera"),
    dict(id=4, tipo="Funcional", descripcion="Webhook sin ejecuciones, workflow inactivo",
         hechos=[("Sintoma", "WebhookSinEjecuciones"), ("Contexto", "WorkflowInactivo")],
         reglas=["R8", "R12"], causas={"WorkflowDesactivado": 0.95}, estado="diagnostico_completo",
         conclusion="WorkflowDesactivado → ActivarElWorkflow"),
    dict(id=5, tipo="Funcional", descripcion="Webhook sin ejecuciones, URL de prueba",
         hechos=[("Sintoma", "WebhookSinEjecuciones"), ("Contexto", "UsaURLTest")],
         reglas=["R9", "R12"], causas={"URLDePruebaEnProduccion": 0.90}, estado="diagnostico_completo",
         conclusion="URLDePruebaEnProduccion → UsarLaURLDeProduccionDelWebhook"),
    dict(id=6, tipo="Funcional", descripcion="Datos vacíos tras un filtro",
         hechos=[("Sintoma", "DatosVacios"), ("Contexto", "FiltroPrevioDescartaTodo")],
         reglas=["R10", "R12"], causas={"FiltroDemasiadoRestrictivo": 0.85}, estado="diagnostico_completo",
         conclusion="FiltroDemasiadoRestrictivo → RevisarCondicionesDelFiltro"),
    dict(id=7, tipo="Funcional", descripcion="401 con API Key",
         hechos=[("ErrorHTTP", 401), ("Autenticacion", "ApiKey")],
         reglas=["R1", "R5", "R12"], causas={"ApiKeyIncorrecta": 0.68}, estado="diagnostico_completo",
         conclusion="ApiKeyIncorrecta → RegenerarYActualizarApiKey"),
    dict(id=8, tipo="Funcional", descripcion="Expresión con campo inexistente",
         hechos=[("Sintoma", "ExpresionInvalida")],
         reglas=["R11", "R12"], causas={"ReferenciaACampoInexistente": 0.80}, estado="diagnostico_completo",
         conclusion="ReferenciaACampoInexistente → CorregirExpresionYVerificarEstructuraJSON"),
    dict(id=9, tipo="Funcional", descripcion="403 con OAuth2 (ampliación)",
         hechos=[("ErrorHTTP", 403), ("Autenticacion", "OAuth2")],
         reglas=["R13", "R14", "R12"], causas={"ScopesInsuficientes": 0.72}, estado="diagnostico_completo",
         conclusion="ScopesInsuficientes → ReautorizarCredencialConScopesCorrectos"),
    dict(id=10, tipo="Funcional", descripcion="Timeout con muchos items (ampliación)",
         hechos=[("Sintoma", "TimeoutNodo"), ("Contexto", "ProcesaMuchosItems")],
         reglas=["R15", "R12"], causas={"TimeoutPorVolumen": 0.80}, estado="diagnostico_completo",
         conclusion="TimeoutPorVolumen → DividirEnLotesConSplitInBatches"),
    # ---------- Caso que activa la mayoría de las reglas ----------
    dict(id=11, tipo="Multifalla", descripcion="Workflow con fallas simultáneas (14 de 15 reglas)",
         hechos=[("ErrorHTTP", 401), ("ErrorHTTP", 403), ("ErrorHTTP", 429), ("ErrorHTTP", 503),
                 ("Autenticacion", "OAuth2"), ("Sintoma", "TimeoutNodo"), ("Contexto", "ProcesaMuchosItems"),
                 ("Sintoma", "WebhookSinEjecuciones"), ("Contexto", "WorkflowInactivo"), ("Contexto", "UsaURLTest"),
                 ("Sintoma", "DatosVacios"), ("Contexto", "FiltroPrevioDescartaTodo"), ("Sintoma", "ExpresionInvalida")],
         reglas=None, causas=TODAS_LAS_CAUSAS_MULTIFALLA, estado="diagnostico_completo",
         conclusion="9 causas raíz, cada una con su recomendación"),
    # ---------- Casos de borde ----------
    dict(id=12, tipo="Sin conclusión", descripcion="Sin hechos de entrada",
         hechos=[], reglas=[], causas={}, estado="sin_conclusion",
         conclusion="Ninguna (entrada vacía)"),
    dict(id=13, tipo="Sin conclusión", descripcion="404: código fuera del alcance",
         hechos=[("ErrorHTTP", 404)], reglas=[], causas={}, estado="sin_conclusion",
         conclusion="Ninguna (ninguna regla aplica)"),
    dict(id=14, tipo="Borde", descripcion="401 sin tipo de autenticación",
         hechos=[("ErrorHTTP", 401)], reglas=["R1"], causas={}, estado="diagnostico_parcial",
         conclusion="Solo CategoriaFalla(CredencialInvalida)"),
    dict(id=15, tipo="Borde", descripcion="503 sin timeout",
         hechos=[("ErrorHTTP", 503)], reglas=["R3"], causas={}, estado="diagnostico_parcial",
         conclusion="Solo CategoriaFalla(ErrorServidorExterno)"),
    dict(id=16, tipo="Borde", descripcion="Hechos duplicados",
         hechos=[("ErrorHTTP", 401), ("ErrorHTTP", 401), ("Autenticacion", "OAuth2")],
         reglas=["R1", "R4", "R12"], causas={"TokenVencido": 0.72}, estado="diagnostico_completo",
         conclusion="TokenVencido (igual que el caso 1)"),
    dict(id=17, tipo="Borde", descripcion="Base con ciclo A→B→A (no debe colgarse)",
         hechos=[("A",)], base=TOY_CICLO, validar=False, ciclo=True,
         reglas=["RA"], causas={}, estado=None,
         conclusion="Termina; detecta el ciclo"),
]


def ejecutar_caso(caso):
    """Ejecuta un caso y compara lo esperado con lo obtenido."""
    r = diagnosticar(caso["hechos"], base_conocimiento=caso.get("base"), validar=caso.get("validar", True))
    reglas_obt = [a["regla"] for a in r["reglas_aplicadas"]]
    causas_obt = {c["causa"]: c["certeza"] for c in r["causas"]}
    checks = {
        "reglas": caso["reglas"] is None or reglas_obt == caso["reglas"],
        "causas": causas_obt == caso["causas"],
        "estado": caso["estado"] is None or r["estado"] == caso["estado"],
        "ciclo": (not caso.get("ciclo")) or bool(r["ciclos_reglas"]),
    }
    return {"resultado": r, "reglas_obtenidas": reglas_obt, "causas_obtenidas": causas_obt,
            "checks": checks, "ok": all(checks.values())}


## 6.2 Ejecución y tabla de validación

> El aviso `WARNING` que aparece en rojo es **esperado**: lo emite el motor al detectar el ciclo A→B→A de la base de juguete del caso 17.

In [7]:
from IPython.display import Markdown, display

def _fmt_hechos(hechos):
    return ", ".join(formatear_hecho(h) for h in hechos) if hechos else "(ninguno)"

def _fmt_resultado(caso, e):
    r = e["resultado"]
    n = len(e["reglas_obtenidas"])
    reglas = " → ".join(e["reglas_obtenidas"]) if n <= 6 else f"{n} reglas ({len(set(e['reglas_obtenidas']))} distintas)"
    if not e["reglas_obtenidas"]:
        return "Sin conclusión"
    if not r["causas"]:
        return f"{reglas} · solo información parcial"
    causas = "; ".join(f"{c['causa']} (CF {c['certeza']:.2f})" for c in r["causas"])
    return f"{reglas} · {causas}" if len(r["causas"]) == 1 else f"{reglas} · {len(r['causas'])} causas"

filas = ["| Caso | Tipo | Descripción | Hechos iniciales | Reglas esperadas | Conclusión esperada | Resultado obtenido | Estado |",
         "|---|---|---|---|---|---|---|---|"]
resultados = {}
for caso in CASOS_PRUEBA:
    e = ejecutar_caso(caso)
    resultados[caso["id"]] = e
    esperadas = "(todas las de las 9 causas)" if caso["reglas"] is None else (" → ".join(caso["reglas"]) or "ninguna")
    hechos_txt = "13 hechos (ver código)" if len(caso["hechos"]) > 8 else _fmt_hechos(caso["hechos"])
    filas.append(f"| {caso['id']} | {caso['tipo']} | {caso['descripcion']} | {hechos_txt} | {esperadas} | "
                 f"{caso['conclusion']} | {_fmt_resultado(caso, e)} | {'✅ Satisfactorio' if e['ok'] else '❌ Falló'} |")

display(Markdown("\n".join(filas)))

total = len(CASOS_PRUEBA)
aprobados = sum(e["ok"] for e in resultados.values())
print(f"\nResultado general: {aprobados} de {total} casos satisfactorios")
assert aprobados == total, "Hay casos que no coinciden con lo esperado"

| Caso | Tipo | Descripción | Hechos iniciales | Reglas esperadas | Conclusión esperada | Resultado obtenido | Estado |
|---|---|---|---|---|---|---|---|
| 1 | Funcional | 401 con OAuth2 | ErrorHTTP(401), Autenticacion(OAuth2) | R1 → R4 → R12 | TokenVencido → ReautenticarCredencialOAuth2 | R1 → R4 → R12 · TokenVencido (CF 0.72) | ✅ Satisfactorio |
| 2 | Funcional | 429 con muchos items | ErrorHTTP(429), Contexto(ProcesaMuchosItems) | R2 → R6 → R12 | RafagaDeSolicitudes → UsarLoopOverItemsConWait | R2 → R6 → R12 · RafagaDeSolicitudes (CF 0.85) | ✅ Satisfactorio |
| 3 | Funcional | 503 con timeout | ErrorHTTP(503), Sintoma(TimeoutNodo) | R3 → R7 → R12 | ServicioExternoSaturado → ActivarRetryOnFailConEspera | R3 → R7 → R12 · ServicioExternoSaturado (CF 0.68) | ✅ Satisfactorio |
| 4 | Funcional | Webhook sin ejecuciones, workflow inactivo | Sintoma(WebhookSinEjecuciones), Contexto(WorkflowInactivo) | R8 → R12 | WorkflowDesactivado → ActivarElWorkflow | R8 → R12 · WorkflowDesactivado (CF 0.95) | ✅ Satisfactorio |
| 5 | Funcional | Webhook sin ejecuciones, URL de prueba | Sintoma(WebhookSinEjecuciones), Contexto(UsaURLTest) | R9 → R12 | URLDePruebaEnProduccion → UsarLaURLDeProduccionDelWebhook | R9 → R12 · URLDePruebaEnProduccion (CF 0.90) | ✅ Satisfactorio |
| 6 | Funcional | Datos vacíos tras un filtro | Sintoma(DatosVacios), Contexto(FiltroPrevioDescartaTodo) | R10 → R12 | FiltroDemasiadoRestrictivo → RevisarCondicionesDelFiltro | R10 → R12 · FiltroDemasiadoRestrictivo (CF 0.85) | ✅ Satisfactorio |
| 7 | Funcional | 401 con API Key | ErrorHTTP(401), Autenticacion(ApiKey) | R1 → R5 → R12 | ApiKeyIncorrecta → RegenerarYActualizarApiKey | R1 → R5 → R12 · ApiKeyIncorrecta (CF 0.68) | ✅ Satisfactorio |
| 8 | Funcional | Expresión con campo inexistente | Sintoma(ExpresionInvalida) | R11 → R12 | ReferenciaACampoInexistente → CorregirExpresionYVerificarEstructuraJSON | R11 → R12 · ReferenciaACampoInexistente (CF 0.80) | ✅ Satisfactorio |
| 9 | Funcional | 403 con OAuth2 (ampliación) | ErrorHTTP(403), Autenticacion(OAuth2) | R13 → R14 → R12 | ScopesInsuficientes → ReautorizarCredencialConScopesCorrectos | R13 → R14 → R12 · ScopesInsuficientes (CF 0.72) | ✅ Satisfactorio |
| 10 | Funcional | Timeout con muchos items (ampliación) | Sintoma(TimeoutNodo), Contexto(ProcesaMuchosItems) | R15 → R12 | TimeoutPorVolumen → DividirEnLotesConSplitInBatches | R15 → R12 · TimeoutPorVolumen (CF 0.80) | ✅ Satisfactorio |
| 11 | Multifalla | Workflow con fallas simultáneas (14 de 15 reglas) | 13 hechos (ver código) | (todas las de las 9 causas) | 9 causas raíz, cada una con su recomendación | 22 reglas (14 distintas) · 9 causas | ✅ Satisfactorio |
| 12 | Sin conclusión | Sin hechos de entrada | (ninguno) | ninguna | Ninguna (entrada vacía) | Sin conclusión | ✅ Satisfactorio |
| 13 | Sin conclusión | 404: código fuera del alcance | ErrorHTTP(404) | ninguna | Ninguna (ninguna regla aplica) | Sin conclusión | ✅ Satisfactorio |
| 14 | Borde | 401 sin tipo de autenticación | ErrorHTTP(401) | R1 | Solo CategoriaFalla(CredencialInvalida) | R1 · solo información parcial | ✅ Satisfactorio |
| 15 | Borde | 503 sin timeout | ErrorHTTP(503) | R3 | Solo CategoriaFalla(ErrorServidorExterno) | R3 · solo información parcial | ✅ Satisfactorio |
| 16 | Borde | Hechos duplicados | ErrorHTTP(401), ErrorHTTP(401), Autenticacion(OAuth2) | R1 → R4 → R12 | TokenVencido (igual que el caso 1) | R1 → R4 → R12 · TokenVencido (CF 0.72) | ✅ Satisfactorio |
| 17 | Borde | Base con ciclo A→B→A (no debe colgarse) | A() | RA | Termina; detecta el ciclo | RA · solo información parcial | ✅ Satisfactorio |


Resultado general: 17 de 17 casos satisfactorios


## 6.3 Cobertura de la base de conocimiento

Qué casos ejercitan cada regla (sin contar el caso 17, que usa una base de juguete):

In [8]:
from collections import defaultdict

cobertura = defaultdict(list)
for caso in CASOS_PRUEBA:
    if caso["id"] == 17:
        continue
    for nombre in set(resultados[caso["id"]]["reglas_obtenidas"]):
        cobertura[nombre].append(caso["id"])

filas = ["| Regla | Descripción | Casos que la ejercitan |", "|---|---|---|"]
for regla in REGLAS:
    casos = sorted(cobertura.get(regla["nombre"], []))
    filas.append(f"| {regla['nombre']} | {regla['descripcion']} | {', '.join(map(str, casos)) if casos else '**ninguno**'} |")
display(Markdown("\n".join(filas)))

sin_cubrir = [r["nombre"] for r in REGLAS if r["nombre"] not in cobertura]
print(f"\nReglas cubiertas: {len(REGLAS) - len(sin_cubrir)} de {len(REGLAS)}")
assert not sin_cubrir, f"Reglas sin caso de prueba: {sin_cubrir}"

| Regla | Descripción | Casos que la ejercitan |
|---|---|---|
| R1 | Un 401 indica credencial inválida | 1, 7, 11, 14, 16 |
| R2 | Un 429 indica límite de solicitudes excedido | 2, 11 |
| R3 | Un código ≥ 500 indica error del servidor externo | 3, 11, 15 |
| R4 | Credencial inválida con OAuth2: token vencido | 1, 11, 16 |
| R5 | Credencial inválida con API Key: clave incorrecta | 7 |
| R6 | Límite excedido con muchos items: ráfaga de solicitudes | 2, 11 |
| R7 | Error del servidor con timeout: servicio saturado | 3, 11 |
| R8 | Webhook sin ejecuciones con workflow inactivo | 4, 11 |
| R9 | Webhook sin ejecuciones con URL de prueba en uso | 5, 11 |
| R10 | Datos vacíos tras un filtro que descarta todo | 6, 11 |
| R11 | Expresión inválida: referencia a un campo inexistente | 8, 11 |
| R12 | Toda causa raíz con solución en el catálogo genera una recomendación | 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 16 |
| R13 | Un 403 indica permisos insuficientes | 9, 11 |
| R14 | Permisos insuficientes con OAuth2: scopes insuficientes | 9, 11 |
| R15 | Timeout con muchos items: lote demasiado grande | 10, 11 |


Reglas cubiertas: 15 de 15


## 6.4 Detalle del caso de fallas simultáneas (caso 11)

El caso 11 muestra cómo el motor deriva varias causas a la vez, cada una con su certeza y su recomendación:

In [9]:
r11 = resultados[11]["resultado"]
print(f"Reglas aplicadas: {len(r11['reglas_aplicadas'])}  |  Reglas distintas: {len(set(a['regla'] for a in r11['reglas_aplicadas']))}  |  Iteraciones: {r11['iteraciones']}\n")
for c in sorted(r11["causas"], key=lambda c: -c["certeza"]):
    print(f"CF {c['certeza']:.2f}  {c['causa']:<30} → {c['recomendacion']}")

Reglas aplicadas: 22  |  Reglas distintas: 14  |  Iteraciones: 3

CF 0.95  WorkflowDesactivado            → ActivarElWorkflow
CF 0.90  URLDePruebaEnProduccion        → UsarLaURLDeProduccionDelWebhook
CF 0.85  RafagaDeSolicitudes            → UsarLoopOverItemsConWait
CF 0.85  FiltroDemasiadoRestrictivo     → RevisarCondicionesDelFiltro
CF 0.80  ReferenciaACampoInexistente    → CorregirExpresionYVerificarEstructuraJSON
CF 0.80  TimeoutPorVolumen              → DividirEnLotesConSplitInBatches
CF 0.72  TokenVencido                   → ReautenticarCredencialOAuth2
CF 0.72  ScopesInsuficientes            → ReautorizarCredencialConScopesCorrectos
CF 0.68  ServicioExternoSaturado        → ActivarRetryOnFailConEspera


## 6.5 Análisis de resultados

- **Todos los casos coinciden con lo esperado.** La validación es automática: se comparan las reglas aplicadas (y su orden), las causas con su factor de certeza y el estado del diagnóstico.
- **Cada regla tiene al menos un caso** que la ejercita (sección 6.3). En la Actividad 4, R5 y R11 quedaban sin probar; ahora tienen los casos 7 y 8.
- **Propagación de certeza:** en el caso 1, R1 (0.90) y R4 (0.80) producen `0.90 × 0.80 = 0.72`, y R12 conserva ese valor al recomendar. En el caso 7, `0.90 × 0.75 = 0.675` se redondea a 0.68.
- **El sistema no inventa conclusiones:** ante un 404 o una entrada vacía responde «sin conclusión» (casos 12 y 13), y ante un 401 sin tipo de autenticación solo clasifica la falla sin afirmar una causa (caso 14).
- **Robustez:** los hechos duplicados no alteran el resultado (caso 16) y una base con un ciclo A→B→A termina sin colgarse, avisando del ciclo (caso 17).

---
# 7. Análisis y conclusiones

## 7.1 Logros

- Una base de conocimiento de **15 reglas** y 8 predicados, formalizada en lógica de predicados y traducida a reglas SI-ENTONCES con factores de certeza.
- Un motor de encadenamiento hacia adelante donde **todas las reglas, incluidas R3 y R12, se tratan de forma uniforme** mediante variables y restricciones, sin atajos.
- Explicación completa del razonamiento y función `porque()` que recorre la cadena hasta los hechos observados.
- Manejo explícito de casos sin conclusión, entradas inválidas y ciclos, con registro del proceso.
- **17 casos de prueba** automáticos que cubren las 15 reglas.
- Un backend que devuelve datos estructurados y no imprime, listo para conectarse a una interfaz.

## 7.2 Limitaciones

- **Sin negación:** el sistema no puede afirmar que algo *no* ocurre. Por eso un 5xx sin timeout solo se clasifica como categoría de falla y no llega a una causa.
- **Un objeto a la vez:** los hechos no indican a qué nodo pertenecen. Si un workflow tiene fallas en nodos distintos, el sistema las trata como un solo conjunto de hechos.
- **Factores de certeza heurísticos:** son juicios de experto, no probabilidades. Si una conclusión pudiera derivarse por dos caminos, el motor conserva el CF de la primera derivación.
- **Cobertura limitada de errores:** 400 y 404 no tienen reglas, y el 403 solo se diagnostica con OAuth2.
- **Hechos ingresados por el usuario:** el sistema no lee el historial de ejecuciones de n8n ni verifica el estado real del workflow.
- Las reglas son heurísticas del dominio y deben revisarse frente a cambios de n8n y de los servicios conectados.

## 7.3 Posibles mejoras

- Reglas para 400, 404 y 403 con API Key.
- Hechos con objeto (`ErrorHTTP(nodo, código)`) para diagnosticar varios nodos en una misma consulta.
- Lectura automática de ejecuciones mediante la API de n8n.
- Combinación de certezas cuando una conclusión se deriva por varios caminos.
- Negación como fallo para poder concluir a partir de la ausencia de un síntoma.

## 7.4 Reflexión sobre el proceso

Al contrastar el notebook de la Actividad 4 con la guía del proyecto final se encontraron brechas concretas: las funciones de explicación imprimían en pantalla en lugar de devolver texto, `porque()` solo mostraba la regla directa, R3 y R12 se resolvían con atajos que no seguían la formalización lógica, y no había casos sin conclusión ni de borde. Separar la lógica de la presentación fue la decisión más importante, porque permite reutilizar el mismo motor en las pruebas y en la interfaz web. Escribir los casos como datos y validarlos automáticamente hizo evidente qué reglas no tenían prueba y facilitará repetir la validación cada vez que se modifique la base de conocimiento.


---
# 8. Estructura del proyecto y siguientes pasos

## 8.1 Estructura de carpetas (con Streamlit)

La guía incluye `templates/` y `static/` solo para Flask; con Streamlit no se necesitan.

```
lopez_narvaez_proyecto_final_sbc/
│
├── README.md                    # Descripción, instalación, cómo ejecutar, link al video
├── requirements.txt             # streamlit (y pandas si se usa)
│
├── src/
│   ├── base_conocimiento.py     # DOMINIOS, PREDICADOS, REGLAS, SOLUCIONES   ← sección 5.1
│   ├── motor_inferencia.py      # motor, explicación, validación, diagnosticar ← sección 5.2
│   ├── casos_prueba.py          # CASOS_PRUEBA y ejecutar_caso (precarga en la interfaz) ← sección 6
│   └── app.py                   # Aplicación Streamlit
│
├── docs/
│   ├── documentacion_tecnica.pdf
│   ├── manual_usuario.pdf
│   └── capturas/                # Capturas de pantalla para el manual
│
└── tests/
    ├── casos_prueba.md          # Tabla de validación documentada
    └── test_motor.py            # (recomendado) ejecuta los 17 casos con pytest
```

## 8.2 Siguientes pasos

1. **Extraer los módulos a `src/`:** copiar las celdas de 5.1, 5.2 y 6.1 a sus archivos y descomentar las líneas de `import` indicadas en cada una. Ejecutar los 17 casos desde `tests/` para confirmar que todo sigue en verde.
2. **Diseñar la interfaz** en papel con las cuatro secciones mínimas: entrada, resultados, explicación y navegación.
3. **Implementar `app.py` en Streamlit** reutilizando `diagnosticar()`, `validar_hechos()`, `PREDICADOS`/`DOMINIOS` y `CASOS_PRUEBA`.
4. **Integrar y probar** el flujo completo con los casos precargados.
5. **Documentar:** documentación técnica y manual de usuario en PDF (con capturas), README y `tests/casos_prueba.md`.
6. **Entrega:** probar en un ambiente limpio, grabar un video de respaldo y comprimir como `lopez_narvaez_proyecto_final_sbc.zip`.
